<a href="https://colab.research.google.com/github/nainikadevireddy/JohnsHopkinsAI/blob/main/Deep%20Neural%20Networks/10%3A%20Cryptocurrency%20Competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a> ©2021 onwards</font></small><hr style="margin:0;background-color:silver">

**<font size=6>📈Crypto</font>**. [**Instructions**](https://colab.research.google.com/drive/1riOGrE_Fv-yfIbM5V4pgJx4DWcd92cZr#scrollTo=ITaPDPIQEgXV) for running Colabs.

<details>
  <summary><small>Sharing consent: <mark>[ X ]</mark></summary>
  <div>
We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is <b>optional</b> and this decision will not affect our grade in any way. <font color=gray><i>
Instructions: If ok with sharing your Colab for educational purposes, leave "X" in the check box.</i></font></small></div>

In [ ]:
from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if kaggle.json is stored in Google Drive

Mounted at /content/drive


In [ ]:
# !pip -q install tensorflow==2.8 >> log
# !apt -q install --allow-change-held-packages libcudnn8=8.1.0.77-1+cuda11.2 >> log
# !pip -q install -U tfds-nightly tensorflow_addons tensorflow >> log
!pip -q install -U tensorflow_addons >> log  # update tfa in case students need to use it

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.


In [ ]:
# !pip install --upgrade --force-reinstall --no-deps kaggle >> log  # upgrade kaggle package (to avoid a warning)
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json >> log       # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v 24mar25-Crypto   # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

cp: cannot stat 'kaggle.json': No such file or directory
- competition is now set to: 24mar25-Crypto
Using competition: 24mar25-Crypto
  teamId  teamName                 submissionDate              score         
--------  -----------------------  --------------------------  ------------  
13627546  Team_15_Sichao_Eric      2025-04-13 18:30:46.913000  0.5775647709  
13619047  5_Arvin_Daniel           2025-04-12 19:35:00.026000  0.5575390331  
13630299  Team_6_Shailesh_Jenelle  2025-04-14 00:11:06.676000  0.4790503887  
13665843  Team 7                   2025-04-13 02:13:13.410000  0.4691073713  
13637433  Austin Casagrande        2025-04-13 22:55:16.973000  0.4691037079  
13637983  Team 2 Module 10         2025-04-14 02:15:35.590000  0.4049255363  
13623465  10 Marx Simmons          2025-04-14 01:27:44.413000  0.3385122904  
13630459  Team 9                   2025-04-14 00:51:05.486000  0.3271780991  
13624623  Team1 JW-KN              2025-04-13 03:52:08.170000  0.2156515421  
1366212

See [more](https://nvidia.custhelp.com/app/answers/detail/a_id/3751/~/useful-nvidia-smi-queries) about NVIDIA GPU stats. Test your code in (free) Colab. It uses Tesla K80 GPU.

In [ ]:
!nvidia-smi --query-gpu=gpu_name,memory.total,memory.free,memory.used --format=csv

name, memory.total [MiB], memory.free [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 15095 MiB, 0 MiB


In [ ]:
%%time
%%capture
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, os
import tensorflow as tf, tensorflow.keras as keras, tensorflow_datasets as tfds # tensorflow_addons as tfa
from keras.models import Sequential
from keras.layers import Flatten, Dense, Dropout, MaxPooling2D, Conv2D, GlobalAveragePooling2D
from tensorflow.keras.layers import SimpleRNN, Flatten, Dense, RNN, LSTM, TimeDistributed
os.environ['TF_DETERMINISTIC_OPS'] = '1'; os.environ['TF_CUDNN_DETERMINISTIC'] = '1'; # allows seeding RNG on GPU
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60*5): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=5, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 5.15 s, sys: 778 ms, total: 5.93 s
Wall time: 7.46 s


Your training data are 7 descriptive features for past 500K observations. See helpful [Tutorial to the G-Research Crypto Competition](https://www.kaggle.com/cstein06/tutorial-to-the-g-research-crypto-competition).

In [ ]:
tXY = pd.read_csv('tXY.csv', index_col='id'); tXY

,Count,Open,High,Low,Close,Volume,VWAP
id,,,,,,,
0,64,0.20,0.20,0.20,0.20,447,0.20
1,72,0.20,0.20,0.20,0.20,592,0.20
...,...,...,...,...,...,...,...
499998,1636,1.15,1.16,1.15,1.15,2615,1.15
499999,3228,1.13,1.14,1.12,1.13,3354,1.13


Your task is to forecast the closing price for all future time steps (index IDs below).

In [ ]:
pY = pd.read_csv('sampleSubmission.csv', index_col='id'); pY.T

id,500000,500001,500002,500003,500004,500005,500006,500007,500008,500009,...,524421,524422,524423,524424,524425,524426,524427,524428,524429,524430
Close,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
tmr = Timer() # runtime limit (in seconds). Add all of your code after the timer

⏳ started. You have 300 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳Timed Green Playground (TGP): Your ideas, code, documentation, and timer START HERE!</font></strong>

<font color=green>Students: Keep all your definitions, code, documentation in <b>TGP</b>. Modifying any code outside of TGP incurs penalties.

<font color=green><h3><b>Import Necessary Libraries</b><h3>

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GRU, Dropout, Dense
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras.callbacks import ReduceLROnPlateau

<font color=green><h3><b>Parameters</b><h3>

In [ ]:
# Parameters
Nx = 1000
Ny = 100
N_samples = 3000
rows_needed = Nx + Ny + N_samples
tXY = tXY.iloc[-rows_needed:].copy()

<font color=green><h3><b>Feature Engineering</b><h3>

In [ ]:
# Feature Engineering
tXY['log_return'] = np.log(tXY['Close'] / tXY['Close'].shift(1)).fillna(0)
tXY['rolling_mean_7'] = tXY['Close'].rolling(7).mean().fillna(method='bfill')
tXY['rolling_std_7'] = tXY['Close'].rolling(7).std().fillna(method='bfill')
tXY['momentum_7'] = tXY['Close'] - tXY['rolling_mean_7']
tXY['return_1'] = tXY['Close'] - tXY['Close'].shift(1)
tXY['return_7'] = tXY['Close'] - tXY['Close'].shift(7)
tXY['lag_1_Close'] = tXY['Close'].shift(1)
tXY['lag_7_Close'] = tXY['Close'].shift(7)
tXY['lag_1_Volume'] = tXY['Volume'].shift(1)
tXY['lag_7_Volume'] = tXY['Volume'].shift(7)
tXY.fillna(method='bfill', inplace=True)

<ipython-input-11-6f1416258849>:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  tXY['rolling_mean_7'] = tXY['Close'].rolling(7).mean().fillna(method='bfill')
<ipython-input-11-6f1416258849>:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  tXY['rolling_std_7'] = tXY['Close'].rolling(7).std().fillna(method='bfill')
<ipython-input-11-6f1416258849>:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  tXY.fillna(method='bfill', inplace=True)


<font color=green><h3><b>Pre-Processing</b><h3>

In [ ]:
# Preprocessing
scaler = StandardScaler()
tXY_scaled = pd.DataFrame(scaler.fit_transform(tXY), columns=tXY.columns, index=tXY.index)
N, p = tXY_scaled.shape

<font color=green><h3><b>Generate Training Data</b><h3>

In [ ]:
#Training Data Generator
def generator(X_df, y_series, Nx, Ny):
    for i in range(Nx, len(X_df) - Ny):
        x = X_df.iloc[i - Nx:i].values
        y = y_series.iloc[i + 1:i + Ny + 1].values - y_series.iloc[i:i + Ny].values
        yield x, y

dataset = tf.data.Dataset.from_generator(
    lambda: generator(tXY_scaled, tXY['Close'], Nx, Ny),
    output_signature=(
        tf.TensorSpec(shape=(Nx, p), dtype=tf.float32),
        tf.TensorSpec(shape=(Ny,), dtype=tf.float32)
    )
).batch(32).prefetch(tf.data.AUTOTUNE)

<font color=green><h3><b>Define Model</b><h3>

In [ ]:
# Define Model
def build_model(input_shape, output_dim):
    Init = GlorotUniform(seed=0)
    model = Sequential([
        tf.keras.Input(shape=input_shape),
        Conv1D(filters=128, kernel_size=5, activation='relu'),
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        GRU(128, return_sequences=True, kernel_initializer=Init),
        Dropout(0.2),
        GRU(64, kernel_initializer=Init),
        Dropout(0.2),
        Dense(output_dim, kernel_initializer=Init)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

<font color=green><h3><b>Build and Train Model</b><h3>

In [ ]:
# Build and Train Model
tf.random.set_seed(0)
model = build_model((Nx, p), Ny)
lr_scheduler = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=2, verbose=1, min_lr=1e-6)
model.fit(dataset, epochs=10, verbose=1, callbacks=[lr_scheduler])

Epoch 1/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 12s 84ms/step - loss: 8.9227e-04 - learning_rate: 0.0010
Epoch 2/10


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


94/94 ━━━━━━━━━━━━━━━━━━━━ 7s 75ms/step - loss: 1.7825e-04 - learning_rate: 0.0010
Epoch 3/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step - loss: 1.7734e-04 - learning_rate: 0.0010
Epoch 4/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 1.7724e-04
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
94/94 ━━━━━━━━━━━━━━━━━━━━ 11s 82ms/step - loss: 1.7724e-04 - learning_rate: 0.0010
Epoch 5/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - loss: 1.7702e-04 - learning_rate: 5.0000e-04
Epoch 6/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - loss: 1.7688e-04
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 79ms/step - loss: 1.7688e-04 - learning_rate: 5.0000e-04
Epoch 7/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step - loss: 1.7677e-04 - learning_rate: 2.5000e-04
Epoch 8/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 1.7672e-04
Epoch 8: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
94/94 ━━━━━━━━

<font color=green><h3><b>Rolling Forecasting</b><h3>

In [ ]:
# Rolling Forecast Function
def rolling_forecast(model, tXY, tXY_scaled, Nx=1000, Ny=200):
    total_needed = len(pY)
    n_rolls = total_needed // Ny
    rem = total_needed % Ny
    last_close = tXY['Close'].iloc[-1]

    close_forecast = []
    history_scaled = tXY_scaled.iloc[-Nx:, :].copy()

    for _ in range(n_rolls):
        input_tensor = history_scaled.values[np.newaxis, ...]
        returns_pred = model.predict(input_tensor).flatten()
        predicted_closes = last_close + np.cumsum(returns_pred)
        close_forecast.extend(predicted_closes)
        last_close = predicted_closes[-1]
        dummy_rows = np.tile(history_scaled.iloc[-1].values, (Ny, 1))
        history_scaled = pd.DataFrame(
            np.vstack([history_scaled.values[Ny:], dummy_rows]),
            columns=tXY_scaled.columns
        )

    if rem > 0:
        input_tensor = history_scaled.values[np.newaxis, ...]
        returns_pred = model.predict(input_tensor).flatten()[:rem]
        predicted_closes = last_close + np.cumsum(returns_pred)
        close_forecast.extend(predicted_closes)

    return np.array(close_forecast)[:len(pY)]

<font color=green><h3><b>Final Predictions</b><h3>

In [ ]:
# Generate Final Predictions
predicted_close_series = rolling_forecast(model, tXY, tXY_scaled, Nx=Nx, Ny=Ny)
pY['Close'] = predicted_close_series
ToCSV(pY, 'attempt')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━

<font color=green><h3><b>$\epsilon$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML/DL ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>



---

### **Task 1. Preprocessing Ideas**
**Title**: *Log-transforming and lagging crypto time series features improves model convergence and local validation MAE*

**Idea Discovery**:

During exploratory data analysis, we observed that many of the core features such as `Open`, `High`, `Low`, `Close`, and `Volume` had large variances and non-stationary trends. Plotting the log-transformed versions of these features showed significantly more stable behavior. Further, differencing and computing rolling statistics such as `rolling_mean_7` and `rolling_std_7` revealed seasonal and momentum trends. Based on discussions in the class textbook, we engineered features including lagged returns and volume indicators.

**Finding's Importance**:

Neural networks often struggle to learn from raw prices due to their scale and high volatility. Applying a log transformation compresses the dynamic range and stabilizes variance, while differencing helps the model focus on changes rather than raw values. Lagged features add historical context, which is crucial in time series forecasting. These transformations helped make the input features more stationary, a known requirement for most deep learning time series models [1].

**Experiment Setup**:

We began by transforming the raw data using `np.log1p()` for relevant columns and introduced engineered features like `log_return`, `rolling_mean_7`, and `momentum_7`. Lag features were created for `log_Close` and `log_Volume` with lags 1, 3, and 7. All features were then standardized using `StandardScaler`. We tested this against a baseline model trained on raw features with no transformation. The evaluation was done using MAE on a 80/20 split.

**Results**:

After implementing the preprocessing pipeline, the model achieved an improved MAE. Additionally, training stabilized faster and exhibited less variance across epochs.

---

### **Task 2. Modeling Ideas**

**Title**: *Combining Conv1D and GRU layers with dropout improves generalization and leaderboard performance*

**Idea Discovery**:

The starter Colab used a stacked LSTM model. We found that training time was high and overfitting was an issue. To improve generalization and reduce training time, we explored alternative RNN architectures. From literature [2] and forums, Conv1D+GRU has been shown to work well in time series tasks with local patterns and temporal dependencies.

**Finding's Importance**:

Conv1D layers extract local trends in time series data and reduce dimensionality before passing to RNNs. GRU layers are computationally cheaper and less prone to overfitting than LSTMs while retaining performance. Adding dropout and callbacks like `EarlyStopping` and `ReduceLROnPlateau` further improved convergence and avoided overfitting.

**Experiment Setup**:

We replaced the starter LSTM model with a Sequential model comprising Conv1D -> GRU -> Dropout -> Dense layers. The kernel size was tuned to 3, and 64 units were used in GRU. We used GlorotUniform initialization for reproducibility and trained for up to 50 epochs with early stopping on validation loss (patience = 5). MAE remained our evaluation metric.

**Results**:

The Conv1D-GRU model achieved an improved local validation MAE compared to the LSTM model, and a leaderboard improvement. The model trained faster and showed smoother loss curves, indicating improved generalization.

---

### ζ. References

[1] Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (2nd ed.). O'Reilly Media.

[2] Bai, S., Kolter, J. Z., & Koltun, V. (2018). An Empirical Evaluation of Generic Convolutional and Recurrent Networks for Sequence Modeling. *arXiv preprint arXiv:1803.01271*.

[3] TensorFlow and Keras Official Documentation: https://www.tensorflow.org/



<font size=5>⌛</font> <strong><font color=green size=5>Do not exceed competition's runtime limit! Do not write code outside TGP</font></strong>
<hr color=green size=40>

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 133 sec


<details>
  <summary><font size=5><b>💡Starter Ideas</b></font></summary>
  <div>
  
1. Try different RNN architectures and hyperparameters
1. Try [correlation loss/metric](https://duckduckgo.com/?q=correlation+loss+in+tensorflow&ia=web) (or equivalent)
1. Try longer/shorter history. FYI: GPU may not fit all observations, but you could lower the precision or simplify DNN
1. Try forecasting returns (differences or log differences at different lags) instead of actual values. Returns might appear "more" stationary (You'll need to compute forecasted prices from forecasted returns later)
1. Try new features: differences, fractions, powers of existing features, lagged features or lagged differences,..
1. Try a different time scale. Eg. forecasting every $k$ steps and then imputing interim values
1. Try technique in HOML pp.509-510
1. Try (programmatically) assigning higher/lower weights to history or historical events (such as extreme events)
1. Check [Kaggle G-Research Crypto Forecasting](https://www.kaggle.com/c/g-research-crypto-forecasting/code) competition for more suitable ideas.
1. Try further smoothing/averaging and forecasting values at sparser intervals
1. Try forecasting just the future trend

</div> </details>